# Experiment 04 — FrozenLake-v1 under DQN: embedding and Data Reuploading

**Attribution first, because it changes what this notebook can claim.** The
dissection paper (arXiv:2511.17112) is **CartPole-only**. FrozenLake appears in
the SimplyQRL *library chapter*, Experiment 3: PPO, 500k steps, **a single seed**,
no classical control, no block-level conclusion. So the repo's usual question —
"does the paper's block finding transfer off-policy?" — **cannot be asked here**.
There is nothing dissected to transfer from. Fig. 6 of the chapter is a sanity
anchor and no more.

**What this experiment is for.** Three things, and the second is the strong one.

1. **Generality of exp03.** exp03 found strong monotone DR transfer under DQN on
   CartPole — greedy 15 → 35 → 199 for L = 1/2/5 at 100k. That is the thesis's
   one clean positive and it rests on a single environment. A DR sweep on a
   structurally different task turns "DR transfers to DQN" into "DR transfers to
   DQN, and not only on CartPole". Failure to replicate bounds the claim, which
   is worth just as much.

2. **FIX-01 where it is measurable.** The phantom fraction is ≈
   `1/mean_episode_length`, so on CartPole it *shrinks as the agent improves* —
   under 1% once an arm learns, which is exactly why exp01 found no significant
   FIX-01 effect in any live arm. On FrozenLake episodes are 6–10 steps whether
   or not the agent is learning. **Measured on this stack: 8.6% (FIX-01 off),
   9.9% (on)** at 6k steps; random-policy prediction 13%.

   And the poison is better aimed. The phantom is `(terminal_state,
   arbitrary_action, r=0, s'=start, done=False)`. Under dense reward that is one
   bad sample among many good ones; here the reward is a single bit delivered only
   at the goal, and the phantom attaches to exactly the transition carrying it.

3. **A controlled version of the chapter's Exp 3.** Config A and Config B, one
   seed each, no control. Re-running them with a capacity-matched control and
   8–10 seeds is a contribution by itself — and at 1 and 4 qubits it is the first
   experiment in this repo where the plan-B robustness pass is affordable.

**Stack stays pinned.** Not because FrozenLake is "in the paper" (it is not), but
because point 2 is a *cross-experiment* comparison against exp01's CartPole null,
and that is only valid if both sides share the stack. See the open question in
`docs/ROADMAP.md`.

**Blocker cleared:** FIX-05. Upstream cannot run a `Discrete` observation space
under DQN at all — see `notebooks/00_fix05_verification.ipynb` and
`docs/CORRECTIONS.md#fix-05`.

---
## 1. Environment

In [ ]:
from google.colab import userdata
GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    GH_TOKEN = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if GH_TOKEN else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import sys, subprocess, pathlib, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp04")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd()/"results"/"exp04"; CODE = pathlib.Path.cwd()
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists(): subprocess.run(["git","-C",str(CODE),"pull","--quiet"], check=False)
    else: subprocess.run(["git","clone","--quiet","-b",BRANCH,REPO_URL,str(CODE)], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(CODE/"requirements.txt")], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(CODE)], check=True)
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","jax","jaxlib"], check=False)

rev = subprocess.run(["git","-C",str(CODE),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip()
print("code:", CODE, "@", rev, " results:", RESULTS)

import sys as _sys
_need=False
if "autoray.autoray" in _sys.modules:
    import autoray.autoray as _aa; _need = not hasattr(_aa,"NumpyMimic")
if "jax" in _sys.modules: _need=True
if _need:
    print("Incompatible modules loaded -> restarting."); os.kill(os.getpid(), 9)
else:
    print("Clean environment: no restart needed. Continue below.")

### After restarting (if it did), run from here

In [ ]:
import sys, pathlib, time, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp04")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd()/"results"/"exp04"; CODE = pathlib.Path.cwd()
sys.path.insert(0, str(CODE/"src"))

import qrl_dissection
from qrl_dissection import analysis, build_arm_config
from qrl_dissection.core import capacity
from qrl_dissection.core.configs import (
    FROZEN_ARM_ENV, FROZEN_DR_A, FROZEN_DR_B, FROZEN_ONEHOT_ID, FROZEN_SCALAR_ID,
)
from qrl_dissection.dqn import GreedyEvalConfig, RunSpec, run_grid

sys.path.insert(0, str(CODE/"experiments"))
from exp04_dqn_frozenlake_embeddings import (
    DQN_KWARGS, STAGE1_ARMS, eval_cfg_for, ladder, random_policy_baseline,
    stage2_arms, summarise,
)
print("ready. Config A depths:", FROZEN_DR_A, " Config B depths:", FROZEN_DR_B)

---
## 2. FIX-05 gate

Not a formality. Without the adapter the replay path fails *silently*: the
built-in transformers return one sample's angles for the whole batch, so a run
completes and produces a plausible flat curve that means nothing.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q",
                    str(CODE/"tests"/"test_frozenlake_envs.py")],
                   capture_output=True, text=True, cwd=str(CODE))
print(r.stdout[-2000:])
assert r.returncode == 0, "FIX-05 gate failed. Do NOT train until this passes."
print("\nFIX-05 gate passed.")

---
## 3. Stage 0 — accounting and liveness

Two numbers decide whether the rest is worth running.

**The capacity ladder.** Every hybrid arm needs a classical control at least its
own size. exp01 was nearly written up with a control that lost on capacity *and*
on information at once; the ladder is what stops that repeating.

**The random-policy baseline.** Its success rate is the floor a learning curve
must clear; its mean episode length gives the predicted phantom fraction. Put
both in `RESULTS-LOG.md` **before** running a grid, so the prediction pre-dates
the measurement.

In [ ]:
ALL_ARMS = STAGE1_ARMS + stage2_arms(FROZEN_DR_A, FROZEN_DR_B, True)
ladder(ALL_ARMS)

base = random_policy_baseline(episodes=2000)
print("\n=== random-policy baseline ===")
print(f"  success rate           {base['success_rate']:.4f}")
print(f"  mean episode length    {base['mean_episode_length']:.2f}")
print(f"  predicted phantom frac {base['predicted_phantom_fraction']:.4f}"
      "   (CartPole at convergence: < 0.01)")

---
## 4. Stage 1 — classical arms and the FIX-01 contrast

Cheap, classical, and where the headline is decided.

| arm | observation | size | role |
|---|---|---|---|
| `frozen_onehot_mlp` | one-hot (16) | oversized | **liveness guard** — if this fails, stop |
| `frozen_scalar_mlp_large` | scalar (1) | oversized | separates encoding from capacity |
| `frozen_matched_scalar` | scalar (1) | matched to Config A | the fair control |

Crossed with FIX-01 on/off, 5 seeds. Classical arms run at roughly 1600 steps/s,
so the whole stage is minutes.

> **What the smoke run already showed** (1 seed, 5k steps): one-hot reaches
> success 0.86–0.96 — the regime is alive and fast. But both *scalar* arms sit at
> 0.03–0.07, barely above random (0.015). A network fed the raw state **index**
> has almost nothing to work with: the ordering is row-major and carries no usable
> metric. So the fair control may be **dead**, which is exactly the situation
> exp01 spent two experiments learning to detect. Read section 6 before drawing
> conclusions from it.

In [ ]:
SEEDS = [1, 2, 3, 4, 5]
STEPS = 100_000
EVERY = 10_000

for env_id, arms in ((FROZEN_ONEHOT_ID, ["frozen_onehot_mlp"]),
                     (FROZEN_SCALAR_ID, ["frozen_scalar_mlp_large", "frozen_matched_scalar"])):
    specs = [RunSpec(arm=a, seed=s, fix_autoreset=fix, total_timesteps=STEPS,
                     dqn_kwargs=DQN_KWARGS)
             for a in arms for fix in (False, True) for s in SEEDS]
    run_grid(specs, RESULTS, env_id=env_id, eval_cfg=eval_cfg_for(env_id, EVERY))

summarise(RESULTS)

### Reading stage 1

FrozenLake's return is a single bit, so a rolling mean **is** the success rate. A
100-episode window rather than the repo's default 50: on a binary signal the
50-window estimator is noisy. `greedy_best` stays primary, as in exp01 — on a
sparse-reward task the gap between it and the ε-greedy training curve is large.

In [ ]:
def success_curve(csv_path, window=100):
    rew, step = analysis.load_episodes(csv_path)
    return step, analysis.moving_average(rew, window)

rows = []
for mp in sorted(RESULTS.glob("*.manifest.json")):
    m = json.loads(mp.read_text())
    if "error" in m: continue
    rows.append((m["spec"]["arm"], m["spec"]["fix_autoreset"], m["outcome"]["episodes_csv"]))

arms = sorted({a for a, _f, _c in rows})
fig, axes = plt.subplots(1, len(arms), figsize=(5*len(arms), 4), sharey=True, squeeze=False)
for ax, arm in zip(axes[0], arms):
    for fix, color in ((False, "tab:red"), (True, "tab:blue")):
        first = True
        for a, f, csvp in rows:
            if a != arm or f != fix or not pathlib.Path(csvp).exists(): continue
            s, y = success_curve(csvp)
            ax.plot(s, y, color=color, alpha=0.6, lw=1.2,
                    label=("FIX-01 on" if fix else "FIX-01 off") if first else None)
            first = False
    ax.axhline(base["success_rate"], ls="--", c="gray", lw=0.8, label="random")
    ax.set_title(arm, fontsize=10); ax.set_xlabel("steps")
axes[0][0].set_ylabel("success rate (MA-100)"); axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.show()

> **Gate before stage 2.** If `frozen_onehot_mlp` has not reached a success rate
> near 1.0, the regime is dead and nothing measured in it is interpretable. Fix
> the regime — longer training, slower ε decay, larger buffer — before spending
> PQC compute.

---
## 5. Stage 2 — the hybrid sweeps

Config A (1 qubit, scalar-to-phase) at DR depth {1, 5, 10, 15} and Config B
(4 qubits, binary-basis) at {1, 5}: the chapter's own grid, run off-policy with
FIX-01 on.

Measured throughput on a CPU runtime: **~50 steps/s at 1 qubit, ~25 at 4**. At
100k steps that is roughly 35 and 65 minutes per run. The probe below re-measures
on your hardware, which is also what tells you whether the 8–10 seed robustness
pass fits in one session.

In [ ]:
t0 = time.time()
run_grid([RunSpec(arm="frozen_binary_4q_L1", seed=99, fix_autoreset=True,
                  total_timesteps=1500, dqn_kwargs=DQN_KWARGS)],
         RESULTS/"_probe", env_id=FROZEN_SCALAR_ID,
         eval_cfg=GreedyEvalConfig(enabled=False))
sps = 1500/(time.time()-t0)
n_cells = len(stage2_arms(FROZEN_DR_A, FROZEN_DR_B, False))
print(f"\n{sps:.0f} steps/s -> {STEPS} steps ~ {STEPS/sps/60:.0f} min per run")
print(f"stage 2 at 3 seeds  ({n_cells*3:2d} cells) ~ {n_cells*3*STEPS/sps/3600:.1f} h")
print(f"stage 2 at 8 seeds  ({n_cells*8:2d} cells) ~ {n_cells*8*STEPS/sps/3600:.1f} h")

In [ ]:
HYB_SEEDS = [1, 2, 3]        # raise to 8-10 for the robustness pass (ROADMAP plan B)
S2 = RESULTS/"stage2"

specs = [RunSpec(arm=a, seed=s, fix_autoreset=True, total_timesteps=STEPS,
                 dqn_kwargs=DQN_KWARGS)
         for a in stage2_arms(FROZEN_DR_A, FROZEN_DR_B, False) for s in HYB_SEEDS]
print(f"{len(specs)} cells")
run_grid(specs, S2, env_id=FROZEN_SCALAR_ID, eval_cfg=eval_cfg_for(FROZEN_SCALAR_ID, EVERY))
summarise(S2)

In [ ]:
rows = []
for mp in sorted(S2.glob("*.manifest.json")):
    m = json.loads(mp.read_text())
    if "error" in m: continue
    arm, oc = m["spec"]["arm"], m["outcome"]
    if not pathlib.Path(oc["episodes_csv"]).exists(): continue
    rew, _ = analysis.load_episodes(oc["episodes_csv"])
    gb = np.nan
    if oc.get("eval_csv") and pathlib.Path(oc["eval_csv"]).exists():
        _, sc = analysis.load_eval(oc["eval_csv"]); gb = float(max(sc)) if len(sc) else np.nan
    rows.append(dict(config="A scalar-to-phase" if "scalar" in arm else "B binary-basis",
                     dr=int(arm.rsplit("_L", 1)[1]), seed=m["spec"]["seed"],
                     best_sr=float(np.nanmax(analysis.moving_average(rew, 100))),
                     greedy=gb))
df2 = pd.DataFrame(rows)
tab = df2.groupby(["config", "dr"]).agg(best_sr=("best_sr","mean"), sd=("best_sr","std"),
                                        greedy=("greedy","mean"), n=("seed","count")).round(3)
display(tab)

fig, ax = plt.subplots(figsize=(7, 4.2))
for cfg, style in (("A scalar-to-phase", "o-"), ("B binary-basis", "s-")):
    if cfg not in tab.index.get_level_values(0): continue
    sub = tab.loc[cfg]
    ax.errorbar(sub.index, sub.best_sr, yerr=sub.sd, fmt=style, capsize=3, label=cfg)
ax.axhline(base["success_rate"], ls=":", c="gray", lw=0.9, label="random")
ax.set_xlabel("Data Reuploading depth (n_layers_q)")
ax.set_ylabel("best success rate (MA-100)")
ax.set_title("FrozenLake-v1 4x4 under DQN: embedding x DR depth")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## 6. The readings, decided in advance

Writing these down before seeing the numbers is what keeps the write-up honest —
the same discipline that turned exp01's disappointing result into a defensible one
rather than a search for a story that fits.

**H1 — DR depth (primary).** Rising then saturating replicates exp03 in a second
environment and generalises the thesis's one clean positive. Flat or falling bounds
that claim to CartPole — a more interesting outcome than a second confirmation.
Falling *with* depth indicates optimisation difficulty, not expressivity, and
should not be called a barren plateau without gradient-norm evidence.

**H2 — embedding.** Config B above Config A at equal depth. Not a transfer claim:
a single-seed uncontrolled Fig. 6 is too weak to transfer from. It is a first
controlled measurement. The `noent` ablation (`--with-ablation`) decides whether
any gap is embedding or entanglement — Config A cannot be entangled at one qubit,
so the two configurations otherwise confound the pair.

**H3 — FIX-01.** From stage 1. A large effect where the phantom fraction is ~10×
CartPole's, against the CartPole null, is a *mechanism* result: the bug's impact
scales with episode turnover, predicted quantitatively before measurement. A null
here too means the bug is real but practically inconsequential — publishable,
honest, and it would substantially reduce the weight the correction can carry.

**H4 — control, and the caveat that may swallow it.** The smoke run suggests both
scalar arms sit near chance. If `frozen_matched_scalar` stays dead, H4 is
**unanswerable**: a hybrid beating a dead control proves nothing, and saying
otherwise would repeat exactly the error exp01 caught. Config A is a 1-qubit
circuit on that same scalar input and may be dead for the same reason — note its
head is `Linear(1, 4)`, so all four Q-values are affine in one expectation value.
If that happens, the informative comparison is **Config B against the one-hot
MLP**, and it must be reported as a result about the *encoding*, not about the
circuit.

**FIX-02 bonus.** Output scaling should be unnecessary here: with a 0/1 terminal
reward and γ = 0.99 the optimal action value is bounded by 1, while PauliZ
expectations already span [−1, 1]. On CartPole the same readout must reach Q ≈ 100.
If the hybrid learns here without scaling, the CartPole scaling requirement is a
*range* problem, not a representational one — one sentence, obtained free.

---

## 7. Closing the loop

Paste the tables into `docs/RESULTS-LOG.md` under Experiment 04, update the
coverage grid in `docs/ROADMAP.md`, then:

```bash
git add docs/ && git commit -m "exp04: FrozenLake under DQN, embedding x DR"
git push && git tag -a exp04-v1 -m "Experiment 04, coverage pass" && git push --tags
```

**Robustness (plan B).** Three seeds is a *coverage* pass and must not become the
final number by default. At 1 and 4 qubits this is the first experiment where 8–10
seeds is genuinely affordable — set `HYB_SEEDS = list(range(1, 11))` and re-run
before writing any of this up as a conclusion. Finished cells are skipped, so the
three you already have count as the first three.